# BirdCLEF 2026 Training v39 — Focal Loss Fine-tune from v30

**Hypothesis**: v30 training used plain BCE which treats every window equally.
Long-tail species (few positive windows) are under-learned.
`FocalLoss(gamma=2)` + inverse-frequency `pos_weight` forces the model to focus on
hard/rare examples.

**What changes from v30:**
- Load v30 weights (not v29)
- Replace `BCEWithLogitsLoss` with `FocalLoss(gamma=2, pos_weight=inv_freq)`
- `pos_weight_c = clip(N_neg_c / N_pos_c, 0.5, 50)` — boosts rare species
- `lr=3e-6` (smaller than v30 5e-6 to avoid forgetting)
- `epochs=12` (more room for focal loss to converge)

**Kaggle inputs required:**
1. `birdclef-2026`
2. `chiragggg/birdclef-2026-perch-embs-v3`
3. `chiragggg/birdclef-2026-perch-weights-v30`


In [ ]:
# === CELL 1: IMPORTS & CONFIG ===
import os, json, copy, random
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR, LinearLR, SequentialLR
from torch.cuda.amp import autocast, GradScaler
from tqdm import tqdm

CFG = dict(
    folds           = 5,
    epochs          = 12,         # more than v30 (10) — focal loss needs more iters
    warmup_epochs   = 1,
    lr              = 3e-6,       # smaller than v30 (5e-6) — fine-tune of fine-tune
    batch_size      = 4,
    num_workers     = 2,
    seed            = 42,
    perch_emb_dim   = 1536,
    perch_emb_noise = 0.005,      # less noise than v30 (0.01)
    gru_hidden      = 512,        # MUST match v23/v30 exactly
    gru_layers      = 2,          # MUST match v23/v30 exactly
    gru_dropout     = 0.3,
    max_seq_len     = 24,
    focal_gamma     = 2.0,        # focal loss exponent
    pos_weight_clip = (0.5, 50.0), # clip inverse-freq weights to [lo, hi]
    checkpoint_tag  = 'v39',
    device          = 'cuda' if torch.cuda.is_available() else 'cpu',
)

random.seed(CFG['seed'])
np.random.seed(CFG['seed'])
torch.manual_seed(CFG['seed'])
device = torch.device(CFG['device'])

print(f"v39 Focal Loss Fine-tuning from v30")
print(f"  Device      : {device}")
print(f"  Epochs      : {CFG['epochs']}  LR={CFG['lr']}")
print(f"  Focal gamma : {CFG['focal_gamma']}")
print(f"  pos_weight  : clipped to {CFG['pos_weight_clip']}")


In [ ]:
# === CELL 2: PATHS & SPECIES ===
def _fe(*candidates):
    return next((p for p in candidates if os.path.exists(p)), candidates[0])

TAXONOMY_CSV    = _fe('/kaggle/input/birdclef-2026/taxonomy.csv',
                      '/kaggle/input/competitions/birdclef-2026/taxonomy.csv')
SOUNDSCAPE_ANNO = _fe('/kaggle/input/birdclef-2026/train_soundscapes_labels.csv',
                      '/kaggle/input/competitions/birdclef-2026/train_soundscapes_labels.csv')

EMBD_DIR = _fe(
    '/kaggle/input/birdclef-2026-perch-embs-v3/perch_embeddings_v3',
    '/kaggle/input/birdclef-2026-perch-embs-v3',
    '/kaggle/input/datasets/chiragggg/birdclef-2026-perch-embs-v3/perch_embeddings_v3',
    '/kaggle/input/datasets/chiragggg/birdclef-2026-perch-embs-v3',
)

# Fine-tune FROM v30
V30_CKPT_DIR = _fe(
    '/kaggle/input/birdclef-2026-perch-weights-v30',
    '/kaggle/input/datasets/chiragggg/birdclef-2026-perch-weights-v30',
)

OUT_DIR = '/kaggle/working'
os.makedirs(OUT_DIR, exist_ok=True)

taxonomy_df = pd.read_csv(TAXONOMY_CSV)
species     = taxonomy_df['primary_label'].astype(str).tolist()
sp_idx      = {lab: i for i, lab in enumerate(species)}
n_classes   = len(species)

_all_emb = list(Path(EMBD_DIR).glob('soundscape_*.npy')) if os.path.isdir(EMBD_DIR) else []
_v30_ckpts = list(Path(V30_CKPT_DIR).glob('perch_gru_v30_fold*.pt')) if os.path.isdir(V30_CKPT_DIR) else []
print(f"Species            : {n_classes}")
print(f"EMBD_DIR           : {EMBD_DIR}  ({len(_all_emb)} files)")
print(f"V30_CKPT_DIR       : {V30_CKPT_DIR}")
print(f"  v30 checkpoints  : {sorted([p.name for p in _v30_ckpts])}")


In [ ]:
# === CELL 3: LABEL HELPERS ===
def soundscape_to_multihot(label_str):
    y = np.zeros(n_classes, dtype='float32')
    for sp in str(label_str).split(';'):
        sp = sp.strip()
        if sp in sp_idx:
            y[sp_idx[sp]] = 1.0
    return y

def _parse_hms(s):
    p = str(s).strip().split(':')
    return int(p[0]) * 3600 + int(p[1]) * 60 + int(p[2])

print('Label helpers defined')


In [ ]:
# === CELL 4: PERCHGRU MODEL (identical architecture to v23/v30) ===
class PerchGRU(nn.Module):
    def __init__(self, n_classes, emb_dim=1536, hidden=512, n_layers=2, dropout=0.3):
        super().__init__()
        self.proj = nn.Sequential(
            nn.LayerNorm(emb_dim),
            nn.Linear(emb_dim, 512),
            nn.GELU(),
        )
        self.gru = nn.GRU(
            512, hidden, n_layers,
            batch_first=True, bidirectional=True,
            dropout=dropout if n_layers > 1 else 0.0,
        )
        self.head = nn.Sequential(
            nn.LayerNorm(hidden * 2),
            nn.Dropout(0.2),
            nn.Linear(hidden * 2, n_classes),
        )

    def forward(self, x):
        single = (x.dim() == 2)
        if single:
            x = x.unsqueeze(1)
        h, _ = self.gru(self.proj(x))
        out   = self.head(h)
        return out.squeeze(1) if single else out


_m = PerchGRU(n_classes).to(device)
_x = torch.randn(2, 8, 1536).to(device)
assert _m(_x).shape == (2, 8, n_classes)
del _m, _x
print(f"PerchGRU OK  (hidden={CFG['gru_hidden']}, layers={CFG['gru_layers']})")


In [ ]:
# === CELL 5: FOCAL LOSS + CLASS-FREQUENCY WEIGHTS ===

class FocalLoss(nn.Module):
    """Binary focal loss with per-class pos_weight.

    FL(p_t) = -alpha_t * (1 - p_t)^gamma * log(p_t)

    We fold pos_weight into the BCEWithLogitsLoss call so the
    focal modulation (1-p_t)^gamma is applied on top.
    """
    def __init__(self, gamma=2.0, pos_weight=None):
        super().__init__()
        self.gamma      = gamma
        self.pos_weight = pos_weight  # (C,) tensor, on same device as logits

    def forward(self, logits, targets):
        # logits, targets: (B, T, C)  or  (B, C)
        with torch.no_grad():
            p    = torch.sigmoid(logits)
            p_t  = p * targets + (1.0 - p) * (1.0 - targets)  # p of true class
            mod  = (1.0 - p_t).pow(self.gamma)                 # focal modulator

        bce = F.binary_cross_entropy_with_logits(
            logits, targets,
            pos_weight=self.pos_weight,
            reduction='none',
        )
        return mod * bce  # (B, T, C) — caller applies mask


# ── Compute inverse-frequency pos_weight from training labels ──────────────
# We need seq_groups for this, but build it inline here so weights are ready
# before the training loop. The full seq_groups is rebuilt in Cell 7.
_sc_anno = pd.read_csv(SOUNDSCAPE_ANNO)
_sc_lmap = {}
for _, row in _sc_anno.iterrows():
    sc_stem  = Path(str(row['filename'])).stem
    end_secs = _parse_hms(row['end'])
    _sc_lmap[(sc_stem, end_secs)] = soundscape_to_multihot(row['primary_label'])

_pos_counts = np.zeros(n_classes, dtype='float64')
_n_windows  = 0
for f in Path(EMBD_DIR).glob('soundscape_*.npy'):
    try:
        stem_part, end_part = f.stem.rsplit('_', 1)
    except ValueError:
        continue
    if not end_part.endswith('s'):
        continue
    end_secs = int(end_part[:-1])
    sc_stem  = stem_part[len('soundscape_'):]
    lv       = _sc_lmap.get((sc_stem, end_secs))
    if lv is None:
        continue
    _pos_counts += lv
    _n_windows  += 1

_neg_counts = _n_windows - _pos_counts
_pw_raw     = _neg_counts / (_pos_counts + 1e-6)
_pw_clipped = np.clip(_pw_raw, CFG['pos_weight_clip'][0], CFG['pos_weight_clip'][1])

_pos_weight_t = torch.from_numpy(_pw_clipped.astype('float32')).to(device)
_criterion    = FocalLoss(gamma=CFG['focal_gamma'], pos_weight=_pos_weight_t)

_active = (_pos_counts > 0).sum()
print(f"Windows total        : {_n_windows}")
print(f"Species with >0 pos  : {_active}/{n_classes}")
print(f"pos_weight  raw range: {_pw_raw.min():.1f} – {_pw_raw.max():.1f}")
print(f"pos_weight clip range: {_pw_clipped.min():.2f} – {_pw_clipped.max():.2f}  (mean={_pw_clipped.mean():.1f})")
print(f"FocalLoss(gamma={CFG['focal_gamma']}) ready")


In [ ]:
# === CELL 6: SOUNDSCAPE SEQUENCE DATASET ===
class SoundscapeSeqDataset(Dataset):
    def __init__(self, seq_groups, emb_root, train=True):
        self.groups   = seq_groups
        self.emb_root = Path(emb_root)
        self.train    = train

    def __len__(self):
        return len(self.groups)

    def __getitem__(self, i):
        grp     = self.groups[i]
        windows = sorted(grp['windows'], key=lambda w: w[1])[:CFG['max_seq_len']]
        T       = len(windows)
        embs    = np.zeros((T, CFG['perch_emb_dim']), dtype='float32')
        labels  = np.zeros((T, n_classes), dtype='float32')
        for t, (stem, end_secs, lv) in enumerate(windows):
            ep = self.emb_root / (stem + '.npy')
            if ep.exists():
                e = np.load(str(ep)).astype('float32')
                if self.train and random.random() < 0.5:
                    e += np.random.randn(*e.shape).astype('float32') * CFG['perch_emb_noise']
                embs[t] = e
            labels[t] = lv
        x = torch.from_numpy(embs)
        y = torch.from_numpy(labels)
        return x, y


def seq_collate(batch):
    xs, ys = zip(*batch)
    max_T  = max(x.shape[0] for x in xs)
    B      = len(xs)
    x_pad  = torch.zeros(B, max_T, CFG['perch_emb_dim'])
    y_pad  = torch.zeros(B, max_T, n_classes)
    mask   = torch.zeros(B, max_T, dtype=torch.bool)
    for i, (x, y) in enumerate(zip(xs, ys)):
        T = x.shape[0]
        x_pad[i, :T] = x
        y_pad[i, :T] = y
        mask[i, :T]  = True
    return x_pad, y_pad, mask


print('SoundscapeSeqDataset + seq_collate defined')


In [ ]:
# === CELL 7: BUILD SOUNDSCAPE SEQUENCE GROUPS ===
sc_anno = pd.read_csv(SOUNDSCAPE_ANNO)
_sc_label_map = {}
for _, row in sc_anno.iterrows():
    sc_stem  = Path(str(row['filename'])).stem
    end_secs = _parse_hms(row['end'])
    lv       = soundscape_to_multihot(row['primary_label'])
    _sc_label_map[(sc_stem, end_secs)] = lv

_sc_groups      = defaultdict(list)
_missing_labels = 0
for f in Path(EMBD_DIR).glob('soundscape_*.npy'):
    try:
        stem_part, end_part = f.stem.rsplit('_', 1)
    except ValueError:
        continue
    if not end_part.endswith('s'):
        continue
    end_secs = int(end_part[:-1])
    sc_stem  = stem_part[len('soundscape_'):]
    lv       = _sc_label_map.get((sc_stem, end_secs))
    if lv is None:
        _missing_labels += 1
        continue
    _sc_groups[sc_stem].append((f.stem, end_secs, lv))

seq_groups = [
    {'stem': s, 'windows': w}
    for s, w in _sc_groups.items()
    if w
]

print(f"Soundscape sequences : {len(seq_groups)}")
print(f"Total windows        : {sum(len(g['windows']) for g in seq_groups)}")
print(f"Missing labels       : {_missing_labels}")
if len(seq_groups) == 0:
    raise RuntimeError("No soundscape sequences found. Check EMBD_DIR.")


In [ ]:
# === CELL 8: 5-FOLD FINE-TUNING FROM v30 WITH FOCAL LOSS ===
print('=' * 65)
print(f"v39 Focal Loss Fine-tuning  folds={CFG['folds']}  AMP={torch.cuda.is_available()}")
print(f"LR={CFG['lr']}  Epochs={CFG['epochs']}  gamma={CFG['focal_gamma']}")
print('=' * 65)

_use_amp = (device.type == 'cuda')

sc_ds = SoundscapeSeqDataset(seq_groups, EMBD_DIR, train=True)
sc_dl = DataLoader(
    sc_ds,
    batch_size=CFG['batch_size'],
    shuffle=True,
    num_workers=CFG['num_workers'],
    collate_fn=seq_collate,
    drop_last=False,
    pin_memory=_use_amp,
)
print(f"DataLoader: {len(sc_ds)} soundscapes  {len(sc_dl)} batches/epoch")

fold_results = []

for fold_idx in range(CFG['folds']):
    v30_ckpt = Path(V30_CKPT_DIR) / f"perch_gru_v30_fold{fold_idx}.pt"
    if not v30_ckpt.exists():
        print(f"[SKIP] v30 checkpoint not found: {v30_ckpt}")
        continue

    print(f"\nFold {fold_idx + 1}/{CFG['folds']}  loading {v30_ckpt.name}")

    model = PerchGRU(
        n_classes, CFG['perch_emb_dim'],
        CFG['gru_hidden'], CFG['gru_layers'], CFG['gru_dropout'],
    ).to(device)
    model.load_state_dict(
        torch.load(v30_ckpt, map_location=device, weights_only=True)
    )
    print("  Loaded v30 weights")

    optimizer = AdamW(model.parameters(), lr=CFG['lr'], weight_decay=1e-4)
    scaler    = GradScaler(enabled=_use_amp)

    warmup_sched = LinearLR(optimizer, start_factor=0.3, end_factor=1.0,
                            total_iters=CFG['warmup_epochs'])
    cosine_sched = CosineAnnealingLR(
        optimizer,
        T_max=max(1, CFG['epochs'] - CFG['warmup_epochs']),
        eta_min=1e-7,
    )
    scheduler = SequentialLR(optimizer,
                              schedulers=[warmup_sched, cosine_sched],
                              milestones=[CFG['warmup_epochs']])

    best_loss  = float('inf')
    best_state = None

    for epoch in range(CFG['epochs']):
        model.train()
        ep_loss   = 0.0
        n_batches = 0

        for x_pad, y_pad, mask in tqdm(sc_dl, desc=f"  Ep {epoch + 1}", leave=False):
            x_pad = x_pad.to(device)
            y_pad = y_pad.to(device)
            mask  = mask.to(device)

            optimizer.zero_grad()
            with autocast(enabled=_use_amp):
                logits  = model(x_pad)              # (B, T, C)
                loss_e  = _criterion(logits, y_pad) # (B, T, C) — focal, unmasked
                m       = mask.unsqueeze(-1).float()
                loss    = (loss_e * m).sum() / m.sum().clamp(min=1)

            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()

            ep_loss   += loss.item()
            n_batches += 1

        ep_loss /= max(n_batches, 1)
        scheduler.step()

        if ep_loss < best_loss:
            best_loss  = ep_loss
            best_state = copy.deepcopy(model.state_dict())

        print(f"  Ep {epoch + 1:2d}/{CFG['epochs']}  focal_loss={ep_loss:.4f}")

    # Restore best checkpoint and save
    if best_state is not None:
        model.load_state_dict(best_state)
    out_ckpt = os.path.join(OUT_DIR, f"perch_gru_v39_fold{fold_idx}.pt")
    torch.save(model.state_dict(), out_ckpt)
    fold_results.append(best_loss)
    print(f"  Saved {out_ckpt}  best_focal_loss={best_loss:.4f}")

    del model, optimizer, scaler
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print(f"\nFine-tuning complete.")
print(f"Saved: {sorted([f for f in os.listdir(OUT_DIR) if 'v39' in f])}")
print(f"Best focal losses per fold: {['%.4f' % l for l in fold_results]}")


In [ ]:
# === CELL 9: UPLOAD AS birdclef-2026-perch-weights-v39 ===
import shutil, subprocess, json as _json

KAGGLE_USERNAME = os.environ.get('KAGGLE_USERNAME', 'chiragggg')
DATASET_SLUG    = 'birdclef-2026-perch-weights-v39'

_upload_dir = '/kaggle/working/upload_v39'
os.makedirs(_upload_dir, exist_ok=True)

_copied = []
for pt in Path(OUT_DIR).glob('perch_gru_v39_fold*.pt'):
    dst = os.path.join(_upload_dir, pt.name)
    shutil.copy2(str(pt), dst)
    _copied.append(pt.name)
print(f"Files to upload: {sorted(_copied)}")

if not _copied:
    print("ERROR: no v39 checkpoints found in", OUT_DIR)
else:
    _meta = {
        'title':    DATASET_SLUG,
        'id':       f'{KAGGLE_USERNAME}/{DATASET_SLUG}',
        'licenses': [{'name': 'CC0-1.0'}],
    }
    with open(os.path.join(_upload_dir, 'dataset-metadata.json'), 'w') as _mf:
        _json.dump(_meta, _mf, indent=2)

    _result = subprocess.run(
        ['kaggle', 'datasets', 'create', '-p', _upload_dir, '--dir-mode', 'zip'],
        capture_output=True, text=True,
    )
    print(_result.stdout)
    if _result.returncode != 0:
        print('STDERR:', _result.stderr)
        print('If dataset already exists, run version instead:')
        print(f'  kaggle datasets version -p {_upload_dir} -m "v39 focal loss"' )
    else:
        print(f'Upload complete: {KAGGLE_USERNAME}/{DATASET_SLUG}')
        print('Attach as birdclef-2026-perch-weights-v39 in inference notebook')
